# Problem Framing and Data Overview

This notebook introduces the enrollment-conditioned deepfake-detection problem and reviews the latest staged dataset used by the repository. It prefers the latest real-data run when one is available.


## Questions This Notebook Answers
- What problem is the repository solving?
- Why are enrollment recordings necessary?
- What do the latest staged manifests look like, and how do they support leakage review?


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
def dataset_mode(run_path: Path) -> str:
    review_path = run_path / 'reports' / 'dataset_review.json'
    if not review_path.exists():
        return 'unknown'
    return json.loads(review_path.read_text()).get('dataset_mode', 'unknown')

fusion_runs = sorted((ROOT / 'outputs' / 'runs').glob('*fusion_plus_interpretable_features'))
real_runs = [run for run in fusion_runs if dataset_mode(run) not in {'demo', 'unknown'}]
selected_run = real_runs[-1] if real_runs else (fusion_runs[-1] if fusion_runs else None)
dataset_review = json.loads((selected_run / 'reports' / 'dataset_review.json').read_text()) if selected_run and (selected_run / 'reports' / 'dataset_review.json').exists() else {}
manifest_root = Path(dataset_review.get('manifest_output_dir', ROOT / 'demo_data' / 'manifests'))
utterances_path = manifest_root / 'utterances.csv'
trials_path = manifest_root / 'trials.csv'
utterances = pd.read_csv(utterances_path) if utterances_path.exists() else pd.DataFrame()
trials = pd.read_csv(trials_path) if trials_path.exists() else pd.DataFrame()
print('Selected run:', selected_run)
print('Manifest root:', manifest_root)
print('Utterances:', len(utterances))
print('Trials:', len(trials))
trials.head()


## Review Notes
Enrollment is the claimed-speaker anchor. The trial manifest makes it possible to inspect whether the probe and enrollment recordings are clearly separated and whether the final task is truly three-way rather than a simpler binary classification problem. When a real run is available, this notebook points at the latest staged real-data manifests so the review starts from the same evidence used in evaluation.
